In [1]:
import json
import numbers
import os
import re
import time
from tabnanny import check

import requests
from numpy import add

mainurl = "https://pokeapi.co/api/v2/"


In [12]:

url = mainurl + "pokemon-species/3"
response = requests.get(url)
print(response.json().keys())


dict_keys(['base_happiness', 'capture_rate', 'color', 'egg_groups', 'evolution_chain', 'evolves_from_species', 'flavor_text_entries', 'form_descriptions', 'forms_switchable', 'gender_rate', 'genera', 'generation', 'growth_rate', 'habitat', 'has_gender_differences', 'hatch_counter', 'id', 'is_baby', 'is_legendary', 'is_mythical', 'name', 'names', 'order', 'pal_park_encounters', 'pokedex_numbers', 'shape', 'varieties'])


In [ ]:
url = mainurl + "ability/1"
response = requests.get(url)
print(response.json()["name"].keys())

In [16]:
# get entried pokemon
filename = "JSON/pokeindex.json"
url = mainurl + "pokedex/36"  # champions dex
response = requests.get(url)
entry_numbers = []
for pokemon in response.json()["pokemon_entries"]:
    entry_numbers.append(int(pokemon["entry_number"]))
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(entry_numbers, indent=4))


In [ ]:
# add megapokemon
filename = "JSON/pokeindex.json"
with open(filename, "r", encoding="utf-8") as f:
    pokeid = json.load(f)
addid = []
for id in pokeid:
    id = int(id)
    url = mainurl + "pokemon-species/" + str(id)
    response = requests.get(url)
    try:
        varieties = response.json()["varieties"]
        for variety in varieties:
            if variety["is_default"] is False:
                url = variety["pokemon"]["url"]
                response = requests.get(url)
                name = response.json()["name"]
                if "mega" in name:
                    print(f"mega exists: {name}")
                    addid.append(int(response.json()["id"]))
    except KeyError:
        print(f"スキップ：ID {id}")
pokeid += addid
with open(filename, "w", encoding="utf-8")as f:
    f.write(json.dumps(pokeid))

In [ ]:
# make pokemon.json
filename = "JSON/pokeindex.json"
with open(filename, "r", encoding="utf-8") as f:
    pokeid = json.load(f)
all_pokemon = {}
for id in pokeid:
    # get info
    id = int(id)
    url = mainurl + "pokemon/" + str(id)
    response = requests.get(url)
    # make case
    abilities = []
    moveids = []
    stats = dict()
    name = response.json()["name"]
    weight = response.json()["weight"]
    height = response.json()["height"]
    for ability in response.json()["abilities"]:
        dc = {}
        dc["ability_id"] = int(ability["ability"]["url"].split("/")[-2])
        dc["ability_name"] = ability["ability"]["name"]
        dc["is_hidden"] = ability["is_hidden"]
        abilities.append(dc)
    for move in response.json()["moves"]:
        version_group_details = move["version_group_details"]
        for version_group_detail in version_group_details:
            if version_group_detail["version_group"]["name"] == "champions":
                moveids.append(int(move["move"]["url"].split("/")[-2]))
    for stat in response.json()["stats"]:
        stats[stat["stat"]["name"]] = stat["base_stat"]
        
    # add gender and jpname

    url = f"{mainurl}pokemon-species/{id}"
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()  # 404エラーなどを検知して例外（except）へ飛ばす
        data = response.json()
        # 性別比率の取得
        gender = data.get("gender_rate")
        jpname = next(
                (
                    name["name"]
                    for name in data.get("names", [])
                    if name["language"]["name"] in ("ja-hrkt", "ja")
                ),
                None,  # 見つからなかった場合のデフォルト値
            )
    except (requests.RequestException, KeyError, ValueError) as e:
        print(f"ID {id} のデータ取得に失敗しました: {e}")
        gender = None
        jpname = None
    all_pokemon[id] = {
        "name": name,
        "jpname": jpname,
        "abilities": abilities,
        "moveids": moveids,
        "stats": stats,
        "gender": gender,
        "weight": weight,
        "height": height
    }
    print(f"{jpname}が終了しました。")
filename = "JSON/pokemon.json"
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(all_pokemon))

In [29]:
import json

filename = "JSON/pokemon.json"

# 1. いったんファイルを読み込む
with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

# 2. ensure_ascii=False を指定して上書き保存する
with open(filename, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("日本語の復元（変換）が完了しました！")

日本語の復元（変換）が完了しました！


In [ ]:
# adding gender and jpname
filename = "pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    all_pokemon = json.load(f)
for id in all_pokemon:
    url = mainurl + "pokemon-species/" + str(id)
    response = requests.get(url)
    gender = response.json()["gender_rate"]
    jpname = response.json()["names"][0]["name"]
    all_pokemon[id]["gender"] = gender
    all_pokemon[id]["jpname"] = jpname
    print(all_pokemon[id]["jpname"])
    
json_string = json.dumps(all_pokemon, ensure_ascii=False)
with open(filename, "w", encoding="utf-8") as f:
    f.write(json_string)

In [ ]:
filename = "JSON/pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    poke = json.load(f)
all_pokemon = {}
url = mainurl + "pokemon/"+str(poke)

In [32]:

# making move.json
filename = "JSON/pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    pokemon_dict = json.load(f)
filename = "JSON/move.json"
with open(filename, "r", encoding="utf-8") as f:
    try:
        move_dict = json.load(f)
    except json.JSONDecodeError:
        # 万が一ファイルが空っぽなどで壊れていた場合の保険
        move_dict = dict()
for id in pokemon_dict:
    for moveid in list(pokemon_dict[id]["moveids"]):
        if str(moveid) not in move_dict:
            url = mainurl + "move/" + str(moveid)
            response = requests.get(url)
            effect_entrys = response.json()["effect_entries"]
            meta = response.json()["meta"]

            for effect_entry in effect_entrys:
                if effect_entry["language"]["name"] == "en":
                    short_effect = effect_entry["short_effect"]
                    break
            stat_changes = response.json()["stat_changes"]
            if len(stat_changes) == 0:
                stat_changes_value = 0
                stat_changes_stat = ""
            else:
                stat_changes_value = stat_changes[0]["change"]
                stat_changes_stat = stat_changes[0]["stat"]["name"]
            names = response.json()["names"]
            for name in names:
                if name["language"]["name"] == "ja-hrkt" or name["language"]["name"] == "ja":
                    jpname = name["name"]
                    break
                else:
                    print(f"{response.json()['name']} name was not found")
                    jpname = ""
            if type(meta) is dict:
                move_dict[str(moveid)] = {
                    "id": moveid,
                    "name": response.json()["name"],
                    "type": response.json()["type"]["name"],
                    "power": response.json()["power"],
                    "accuracy": response.json()["accuracy"],
                    "pp": response.json()["pp"],
                    "damage_class": response.json()["damage_class"]["name"],
                    "effect": short_effect,
                    "effect_chance": response.json()["effect_chance"],
                    "stat_changes_value": stat_changes_value,
                    "stat_changes_stat": stat_changes_stat,
                    "jpname": jpname,
                    # meta
                    "ailment": meta["ailment"]["name"],
                    "ailment_chance": meta["ailment_chance"],
                    "category": meta["category"]["name"],
                    "crit_rate": meta["crit_rate"],
                    "flinch_chance": meta["flinch_chance"],
                    "healing": meta["healing"],
                    "min_hits": meta["min_hits"],
                    "max_hits": meta["max_hits"],
                    "min_turns": meta["min_turns"],
                    "max_turns": meta["max_turns"],
                    "stat_chance": meta["stat_chance"],
                    "drain": meta["drain"],
                }
                print(f"追加：ID {moveid} - {move_dict[str(moveid)]['name']} exist meta")
            else:
                move_dict[str(moveid)] = {
                    "id": moveid,
                    "name": response.json()["name"],
                    "type": response.json()["type"]["name"],
                    "power": response.json()["power"],
                    "accuracy": response.json()["accuracy"],
                    "pp": response.json()["pp"],
                    "damage_class": response.json()["damage_class"]["name"],
                    "effect": short_effect,
                    "effect_chance": response.json()["effect_chance"],
                    "stat_changes_value": stat_changes_value,
                    "stat_changes_stat": stat_changes_stat,
                    "jpname": jpname,
                }
                print(f"追加：ID {moveid} - {move_dict[str(moveid)]['name']} except meta")
        else:
            print(f"スキップ：ID {moveid}")

        # save
    with open(filename, "w", encoding="utf-8") as f:
        f.write(json.dumps(move_dict, indent=4))
    print(f"finish {pokemon_dict[id]["name"]}")


スキップ：ID 14
スキップ：ID 34
スキップ：ID 38
スキップ：ID 46
スキップ：ID 63
スキップ：ID 73
スキップ：ID 74
スキップ：ID 76
スキップ：ID 77
スキップ：ID 79
スキップ：ID 80
スキップ：ID 89
スキップ：ID 92
スキップ：ID 113
スキップ：ID 133
スキップ：ID 156
スキップ：ID 164
スキップ：ID 173
スキップ：ID 174
スキップ：ID 182
スキップ：ID 184
スキップ：ID 188
スキップ：ID 200
スキップ：ID 202
スキップ：ID 203
スキップ：ID 204
スキップ：ID 214
スキップ：ID 230
スキップ：ID 235
スキップ：ID 241
スキップ：ID 263
スキップ：ID 270
スキップ：ID 275
スキップ：ID 282
スキップ：ID 311
スキップ：ID 331
スキップ：ID 338
スキップ：ID 388
スキップ：ID 398
スキップ：ID 402
スキップ：ID 412
スキップ：ID 414
スキップ：ID 416
スキップ：ID 437
スキップ：ID 438
スキップ：ID 447
スキップ：ID 474
スキップ：ID 482
スキップ：ID 491
スキップ：ID 496
スキップ：ID 523
スキップ：ID 572
スキップ：ID 580
スキップ：ID 707
スキップ：ID 803
スキップ：ID 805
スキップ：ID 885
finish venusaur
スキップ：ID 7
スキップ：ID 9
スキップ：ID 14
スキップ：ID 19
スキップ：ID 25
スキップ：ID 34
スキップ：ID 38
スキップ：ID 44
スキップ：ID 46
スキップ：ID 53
スキップ：ID 63
スキップ：ID 68
スキップ：ID 76
スキップ：ID 83
スキップ：ID 89
スキップ：ID 91
スキップ：ID 126
スキップ：ID 156
スキップ：ID 157
スキップ：ID 164
スキップ：ID 173
スキップ：ID 182
スキップ：ID 184
スキップ：ID 187
スキップ：ID 200
スキップ：ID 201
スキップ：ID 203
スキップ：ID

In [4]:
# sorting
filename = "JSON/ability.json"
with open(filename, "r", encoding="utf-8") as f:
    move_dict = json.load(f)
move_dict = dict(sorted(move_dict.items(), key=lambda item: int(item[0])))
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(move_dict, indent=4, ensure_ascii=False))

In [ ]:

# adding weight
filename = "pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    pokemon_dict = json.load(f)
for id in pokemon_dict:
    id = int(id)
    url = mainurl + "pokemon/" + str(id)
    response = requests.get(url)
    weight = response.json()["weight"]
    height = response.json()["height"]
    pokemon_dict[str(id)]["weight"] = weight
    pokemon_dict[str(id)]["height"] = height
    print(f"追加：ID {id} - {pokemon_dict[str(id)]['name']}")
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(pokemon_dict, indent=4, ensure_ascii=False))

In [ ]:
# making ability.json
filename = "pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    pokemon_dict = json.load(f)

filename = "ability.json"
with open(filename, "r", encoding="utf-8") as f:
    try:
        ability_dict = json.load(f)
    except json.JSONDecodeError:
        ability_dict = {}
for id in pokemon_dict:
    abilities = pokemon_dict[id]["abilities"]  
    for ability in abilities:
        id = int(ability["ability_id"])
        url = mainurl + "ability/" + str(id)
        response = requests.get(url)
        effect_entries = response.json()["effect_entries"]
        for effect_entry in effect_entries:
            if effect_entry["language"]["name"] == "en":
                effect = effect_entry["effect"]
                short_effect = effect_entry["short_effect"]
                break
        natural_name = response.json()["name"]
        names = response.json()["names"]
        for name in names:
            if name["language"]["name"] == "ja-hrkt":
                jpname = name["name"]
                break
            elif name["language"]["name"] == "ja":
                jpname = name["name"]
                break
            else:
                jpname = ""
        ability_dict[id] = {
            "effect": effect,
            "short_effect": short_effect,
            "name": natural_name,
            "jpname": jpname
        }
        print(f"add {jpname},{natural_name}{effect}")
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(ability_dict, indent=4, ensure_ascii=False))

add しんりょく,overgrowWhen this Pokémon has 1/3 or less of its HP remaining, its Grass-type moves inflict 1.5× as much regular damage.
add ようりょくそ,chlorophyllThis Pokémon's Speed is doubled during strong sunlight.

This bonus does not count as a stat modifier.
add もうか,blazeWhen this Pokémon has 1/3 or less of its HP remaining, its Fire-type moves inflict 1.5× as much regular damage.
add サンパワー,solar-powerDuring strong sunlight, this Pokémon has 1.5× its Special Attack but takes 1/8 of its maximum HP in damage after each turn.
add げきりゅう,torrentWhen this Pokémon has 1/3 or less of its HP remaining, its Water-type moves inflict 1.5× as much regular damage.
add あめうけざら,rain-dishThis Pokémon heals for 1/16 of its maximum HP after each turn during rain.
add むしのしらせ,swarmWhen this Pokémon has 1/3 or less of its HP remaining, its Bug-type moves inflict 1.5× as much regular damage.

Overworld: If the lead Pokémon has this ability, the wild encounter rate is increased.
add スナイパー,sniperThis Pokémon infli